In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_cohere import ChatCohere
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [12]:
load_dotenv()
model = ChatCohere(model="command-r7b-12-2024")

In [21]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal['Negative','Positive']
    diagnosis : dict
    result : str

class SentimentSchema(BaseModel):
    sentiment : Literal['Positive','Negative']

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

structured_model1 = llm.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

def find_sentiment(state : ReviewState) -> Literal['Positive','Negative']:
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model1.invoke(prompt).sentiment

    return {'sentiment': sentiment}

def run_diagnosis(state : ReviewState):
    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
    """
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

def positive(state : ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
    Also, kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content

    return {'response': response}

def check_sentiment(state : ReviewState) -> Literal["positive", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive'
    else:
        return 'run_diagnosis'

In [23]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('positive',positive)
graph.add_node('negative',negative)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive',END)
graph.add_edge('run_diagnosis','negative')
graph.add_edge('negative',END)

workflow = graph.compile()

inital_state = {'review':"Bad product"}
workflow.invoke(inital_state)

{'review': 'Bad product',
 'sentiment': 'Negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'}}